# Chapitre 6 - Cours 3 : points chauds et routage des requêtes

Les cours 1 et 2 ont réparti des **clés** équitablement. Ce notebook aborde deux
questions qui restent : que faire quand quelques clés reçoivent l'essentiel des
**requêtes** (le compte d'une célébrité, un message devenu viral) ? Et comment une
requête trouve-t-elle la partition, donc le nœud, qui détient sa
clé, alors même que les morceaux se déplacent ? Il couvre les sections *Skewed Workloads
and Relieving Hot Spots* et *Request Routing* (avec *Parallel Query Execution*).

**Plan.**

1. Le hachage répartit les clés, pas la charge : requêtes en loi de Zipf.
2. La célébrité : une clé chaude qui dure quelques heures.
3. Saler les clés chaudes : éclater les écritures, payer à la lecture.
4. Isoler une clé chaude, et distinguer lectures et écritures.
5. Trois façons de router une requête.
6. Qui détient la carte ? Coordinateur, notifications et diffusion épidémique.
7. Les requêtes qui touchent tous les morceaux : exécution parallèle.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import partitioning as P
from ddia.diagrams import Diagram
from ddia.timelines import SpaceTime
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET, COLORS,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
rng = np.random.default_rng(0)
fmt = lambda v: f"{v:,.0f}".replace(",", " ")